# Step 6: statistical tests and figures

Runs the statistical tests (thesis Section 3.7 and Table 3) and draws the result figures with plotnine, from the node
pairs saved by `05_compare`. Saves the figures, `tests.csv` and the normality check to `runs/figures/`.

Settings.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup messages

PROJECT = os.path.expandvars("$SCRATCH/CSC-488-Code")    # project folder: data, lab code and results
DATA_PATH = os.path.join(PROJECT, "data", "Copy of ecoli_MG1655plus_genome_part2_log_counts_norm_01.csv")
SEQADAGE = os.path.join(PROJECT, "seqADAGE")              # the lab's code (my fork, Jordin branch)
RUN_DIR = os.path.join(PROJECT, "runs")                   # every notebook saves and reads its results here
os.chdir(PROJECT)

import textwrap
import numpy as np
import pandas as pd
from scipy.stats import shapiro, ttest_rel
from plotnine import (ggplot, aes, geom_boxplot, geom_jitter, geom_violin, geom_point, geom_segment, geom_text,
                      facet_wrap, labs, theme_classic, theme, element_text, element_blank, scale_fill_manual,
                      scale_color_manual, scale_x_continuous, scale_y_continuous, coord_cartesian)
from mizani.breaks import breaks_extended

FIG_DIR = os.path.join(RUN_DIR, "figures")
DPI = 300                        # resolution of the saved figures
BASE_SIZE = 18                   # font size
N_POINTS = 1500                  # node pairs drawn as dots per box
STEP_MP, STEP_NP = 0.025, 0.08   # spacing of the significance brackets (model-pair and node-pair plots)

Load the node pairs, and take the median per pair of models.

In [ ]:
GRAY, LIGHT_GRAY, BLUE, ORANGE = "#8a8f98", "#c3c2b7", "#2a78d6", "#eb6834"


def load_node_pairs():
    """Every compared pair of nodes, as saved by 05_compare."""
    cols = ["comparison", "group", "group_b", "seed_a", "seed_b", "keep", "shap_spearman"]
    nodes = pd.read_csv(os.path.join(RUN_DIR, "results", "node_pairs.csv"), usecols=cols)
    print(f"{len(nodes):,} node pairs")
    return nodes


def per_pair_medians(nodes, comparisons):
    """One value per pair of models: the median agreement over its node pairs."""
    d = nodes[nodes["comparison"].isin(comparisons)]
    keys = ["comparison", "group", "group_b", "seed_a", "seed_b"]
    return d.groupby(keys, dropna=False)["shap_spearman"].median().reset_index()

The statistical test. Each independent unit (a group, or a pair of groups) gets one value, the median over its pairs
of models. The differences between two conditions are checked with a Shapiro-Wilk test and tested with a two-sided
paired t-test.

In [ ]:
def stars(p):
    for cutoff, label in [(1e-4, "****"), (1e-3, "***"), (1e-2, "**"), (0.05, "*")]:
        if p < cutoff:
            return label
    return "ns"


def unit_values(per_pair, comparison):
    """One value per group (within-group comparisons) or per pair of groups (across-group comparisons)."""
    d = per_pair[per_pair["comparison"] == comparison]
    keys = ["group"] if d["group_b"].isna().all() else ["group", "group_b"]
    return d.groupby(keys)["shap_spearman"].median()


def paired_test(a, b, comparison):
    """Is condition b different from condition a, within the same units?"""
    a, b = a.align(b, join="inner")
    diff = (b - a).to_numpy(float)
    p = ttest_rel(b, a).pvalue
    units = "pairs of groups" if isinstance(a.index, pd.MultiIndex) else "groups"
    return dict(comparison=comparison, units=f"{len(diff)} {units}", median_a=float(np.median(a)),
                median_b=float(np.median(b)), median_difference=float(np.median(diff)),
                smallest_difference=diff.min(), largest_difference=diff.max(), units_b_higher=int((diff > 0).sum()),
                shapiro_p=shapiro(diff).pvalue, p=p, stars=stars(p), differences=diff)

Plot helpers: box plots with dots and significance brackets, and the violin plots for the normality check.

In [ ]:
def node_sample(nodes, comparisons, n_points, seed=0):
    """A random sample of node pairs per comparison, drawn as dots."""
    d = nodes[nodes["comparison"].isin(comparisons)]
    return pd.concat([g.sample(min(n_points, len(g)), random_state=seed) for _, g in d.groupby("comparison")])


def whisker_ends(data, end="top"):
    """End of each box's upper (or lower) whisker: the most extreme value within 1.5 box-lengths of the box."""
    rows = []
    for box, v in data.groupby("box")["value"]:
        q1, q3 = v.quantile([0.25, 0.75])
        inside = v[(v <= q3 + 1.5 * (q3 - q1)) & (v >= q1 - 1.5 * (q3 - q1))]
        rows.append(dict(box=box, value=inside.max() if end == "top" else inside.min()))
    return pd.DataFrame(rows)


def bracket_layers(brackets, size, span):
    """A line between two boxes with short ends, and the stars above it."""
    b = brackets.assign(y_end=brackets["y"] - 0.015 * span)
    return [geom_segment(aes(x="x1", xend="x2", y="y", yend="y"), data=b, inherit_aes=False, size=0.7),
            geom_segment(aes(x="x1", xend="x1", y="y", yend="y_end"), data=b, inherit_aes=False, size=0.7),
            geom_segment(aes(x="x2", xend="x2", y="y", yend="y_end"), data=b, inherit_aes=False, size=0.7),
            geom_text(aes(x="(x1 + x2) / 2", y="y", label="label"), data=b, inherit_aes=False, va="bottom",
                      size=size, nudge_y=0.004 * span)]


def stack_brackets(pairs, data, positions, step):
    """Place each bracket just above the boxes it spans, and above any earlier bracket it overlaps."""
    x_of = data["box"].map(positions)
    rows = []
    for a, b, label in pairs:
        x1, x2 = sorted((positions[a], positions[b]))
        y = data.loc[(x_of >= x1) & (x_of <= x2), "value"].max() + step
        for r in rows:
            if r["x1"] <= x2 and x1 <= r["x2"]:
                y = max(y, r["y"] + step)
        rows.append(dict(x1=x1, x2=x2, y=y, label=label))
    return pd.DataFrame(rows)


def box_plot(data, positions, labels, colors, brackets, ylabel, title, facet=None, point_size=1.6, points=None,
             ylim=None):
    """Box plot (box = middle half, line = median, whiskers = 1.5 box-lengths) with dots and brackets."""
    d = data.assign(x=data["box"].map(positions))
    pts = d if points is None else points.assign(x=points["box"].map(positions))
    alpha = float(np.clip(8 / np.sqrt(pts.groupby("box").size().max()), 0.12, 0.8))  # many dots -> see-through
    p = (ggplot(d, aes("x", "value", group="box"))
         + geom_jitter(aes(color="box"), data=pts, width=0.16, height=0, size=point_size, alpha=alpha,
                       random_state=1)
         + geom_boxplot(aes(fill="box"), width=0.55, outlier_shape="", alpha=0.25, size=0.9)
         + scale_fill_manual(values=colors, guide=None)
         + scale_color_manual(values=colors, guide=None)
         + scale_x_continuous(breaks=[positions[k] for k in labels], labels=list(labels.values()), expand=(0, 0.5))
         + scale_y_continuous(breaks=lambda lims: [b for b in breaks_extended(n=5)(lims) if b <= 1])
         + labs(x="", y=ylabel, title=title)
         + theme_classic(base_size=BASE_SIZE)
         + theme(plot_title=element_text(size=BASE_SIZE + 2), axis_text_x=element_text(size=BASE_SIZE - 2)))
    lo, hi = ylim if ylim else (min(d["value"].min(), pts["value"].min()), brackets["y"].max())
    for layer in bracket_layers(brackets, size=BASE_SIZE * 0.9, span=hi - lo):
        p += layer
    if facet:
        p += facet_wrap(f"~{facet}")
    if ylim:
        p += coord_cartesian(ylim=ylim)
    return p


def violin_check(checks):
    """Violin plot of the paired differences behind each test, with the Shapiro-Wilk p-value."""
    rows, labels = [], {}
    for name, values in checks.items():
        p = shapiro(values).pvalue
        labels[name] = textwrap.fill(name, 24) + f"\nn = {len(values)}, p = {p:.2g}"
        rows.append(pd.DataFrame({"set": name, "value": values}))
    d = pd.concat(rows, ignore_index=True)
    d["set"] = pd.Categorical(d["set"].map(labels), categories=list(labels.values()))
    return (ggplot(d, aes(x=0, y="value"))
            + geom_violin(fill="#cfd8e8", color="#4a5058", size=0.6)
            + geom_boxplot(width=0.08, outlier_shape="", fill="white", size=0.5)
            + geom_point(color=BLUE, size=2.5, alpha=0.8)
            + facet_wrap("~set", nrow=1, scales="free_y")
            + labs(x="", y="difference within each unit",
                   title="Normality check of the paired differences: Shapiro-Wilk p (p < 0.05 = not normal)")
            + theme_classic(base_size=16)
            + theme(axis_text_x=element_blank(), axis_ticks_major_x=element_blank(),
                    strip_text=element_text(size=10), plot_title=element_text(size=16), panel_spacing=0.04))

Make one figure: run its tests, then draw it with dots per node pair (`node_pairs`) and/or per pair of models
(`model_pairs`). Each bracket shows the stars and the median difference.

In [ ]:
YLABEL = {"model_pairs": "Agreement of SHAP explanations\n(median Spearman per pair of models)",
          "node_pairs": "Agreement of SHAP explanations\n(Spearman per pair of nodes)"}


def make_figure(name, comparisons, rename, positions, labels, colors, pairs, title, width, height, versions,
                subsets=None, facet=None, point_size=2.4):
    """pairs: (comparison a, comparison b, description), one per bracket. subsets: [(panel title, node pairs)]."""
    subsets = subsets or [(None, nodes)]
    parts = {k: [] for k in ["model_pairs", "node_pairs", "node_points", "brackets_mp", "brackets_np", "bottoms"]}
    for subset, nd in subsets:
        pp = per_pair_medians(nd, comparisons)
        tag = {} if subset is None else {facet: subset}
        mp = pd.DataFrame({"box": pp["comparison"].map(rename), "value": pp["shap_spearman"]})
        sel = nd[nd["comparison"].isin(comparisons)]
        npd = pd.DataFrame({"box": sel["comparison"].map(rename), "value": sel["shap_spearman"]})
        smp = node_sample(nd, comparisons, N_POINTS)
        pts = pd.DataFrame({"box": smp["comparison"].map(rename), "value": smp["shap_spearman"]})
        bracket_pairs = []
        for a, b, what in pairs:
            label = what + (f" ({subset.lower()})" if subset not in (None, "All nodes") else "")
            r = paired_test(unit_values(pp, a), unit_values(pp, b), label)
            diff = r.pop("differences")
            if label not in checks:  # the same test can label two figures
                checks[label] = diff
                tests.append(r)
            bracket_pairs.append((rename[a], rename[b], f"{r['stars']} ({r['median_difference']:+.2f})"))
        parts["model_pairs"].append(mp.assign(**tag))
        parts["node_pairs"].append(npd.assign(**tag))
        parts["node_points"].append(pts.assign(**tag))
        parts["brackets_mp"].append(stack_brackets(bracket_pairs, mp, positions, STEP_MP).assign(**tag))
        tops = pd.concat([pts, whisker_ends(npd, "top")], ignore_index=True)
        parts["brackets_np"].append(stack_brackets(bracket_pairs, tops, positions, STEP_NP).assign(**tag))
        parts["bottoms"].append(min(pts["value"].min(), whisker_ends(npd, "bottom")["value"].min()))
    cat = lambda k: pd.concat(parts[k], ignore_index=True)
    node_ylim = (min(parts["bottoms"]) - 0.03, cat("brackets_np")["y"].max() + STEP_NP)
    drawn = {"model_pairs": (cat("model_pairs"), None, cat("brackets_mp"), point_size, None),
             "node_pairs": (cat("node_pairs"), cat("node_points"), cat("brackets_np"), 1.2, node_ylim)}
    figs = {}
    for version in versions:
        data, points, brackets, size, ylim = drawn[version]
        p = box_plot(data, positions, labels, colors, brackets, YLABEL[version], title,
                     facet=facet if subsets[0][0] else None, point_size=size, points=points, ylim=ylim)
        p.save(os.path.join(FIG_DIR, f"{name}_{version}.png"), width=width, height=height, dpi=DPI, verbose=False)
        figs[version] = p
    return figs

Run the tests and draw the thesis figures: inside one group, across runs, and with and without quiet nodes.

In [ ]:
nodes = load_node_pairs()
os.makedirs(FIG_DIR, exist_ok=True)
tests, checks = [], {}

within = make_figure(
    "within_group", ["random_pairs", "ind_vs_ind"], {"random_pairs": "random", "ind_vs_ind": "same"},
    {"random": 1, "same": 2}, {"random": "Random pairs\n(chance level)", "same": "Same cluster"},
    {"random": GRAY, "same": BLUE}, [("random_pairs", "ind_vs_ind", "within groups: random vs same cluster")],
    "Inside one group", width=7.5, height=7, versions=["node_pairs"], point_size=1.6)

rename = {"across_ind_random": "ind_random", "across_ind": "ind", "across_eadage_random": "ead_random",
          "across_eadage": "ead"}
across = make_figure(
    "across_runs", list(rename), rename, {"ind_random": 1, "ind": 2, "ead_random": 3.4, "ead": 4.4},
    {"ind_random": "Single models\nrandom node", "ind": "Single models\nmatched",
     "ead_random": "eADAGEs\nrandom node", "ead": "eADAGEs\nmatched"},
    {"ind_random": GRAY, "ind": BLUE, "ead_random": LIGHT_GRAY, "ead": ORANGE},
    [("across_ind_random", "across_ind", "single models: random vs matched node"),
     ("across_eadage_random", "across_eadage", "eADAGEs: random vs matched node"),
     ("across_ind", "across_eadage", "matched nodes: single models vs eADAGEs")],
    "Across independent runs", width=9.5, height=7.5, versions=["node_pairs", "model_pairs"])

quiet = make_figure(
    "quiet_nodes", ["across_ind", "across_eadage"], {"across_ind": "ind", "across_eadage": "ead"},
    {"ind": 1, "ead": 2}, {"ind": "Single\nmodels", "ead": "eADAGEs"}, {"ind": BLUE, "ead": ORANGE},
    [("across_ind", "across_eadage", "matched nodes: single models vs eADAGEs")],
    "Across runs, with and without quiet nodes", width=9.5, height=7, versions=["node_pairs"],
    subsets=[("All nodes", nodes), ("Without quiet nodes", nodes[nodes["keep"]])], facet="subset")

order = ["matched nodes: single models vs eADAGEs", "matched nodes: single models vs eADAGEs (without quiet nodes)",
         "eADAGEs: random vs matched node", "single models: random vs matched node",
         "within groups: random vs same cluster"]  # as in Table 3, main result first
tests = pd.DataFrame(tests).set_index("comparison").loc[order].reset_index()
tests.to_csv(os.path.join(FIG_DIR, "tests.csv"), index=False)
normality = violin_check(checks)
normality.save(os.path.join(FIG_DIR, "normality_check.png"), width=16, height=7, dpi=150, verbose=False)
print("saved to", FIG_DIR)

The statistical tests (thesis Table 3).

In [ ]:
tests.round(4)

The figures and the normality check.

In [ ]:
for fig in [within, across, quiet]:
    for p in fig.values():
        display(p)
display(normality)